## EDGAR Extraction Pipeline

**Goal:** Download 10-K annual reports for the companies selected in `01_explore_companies.ipynb` and extract the **Item 1A (Risk Factors)** section from each filing.

**Run cells top to bottom — no skipping.**

## 1. Imports & Configuration

In [1]:
import re
import time
import warnings
import requests
import pandas as pd
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from tqdm import tqdm
from pathlib import Path

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

HEADERS = {
    "User-Agent": "daphne s_hsueh25@stud.hwr-berlin.de",
    "Accept-Encoding": "gzip, deflate",
}

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)
(DATA_DIR / "raw_filings").mkdir(exist_ok=True)
(DATA_DIR / "item1a").mkdir(exist_ok=True)

REQUEST_DELAY = 0.15

START_YEAR = 2010
END_YEAR   = 2026

# Set to a small number (e.g. 3) for a quick test run; None to process all
MAX_COMPANIES = None

print("Configuration ready.")
print(f"Saving data to: {DATA_DIR.resolve()}")

Configuration ready.
Saving data to: /Users/hdaphne/Desktop/bipm/NLP/topic-modeling-G5/data


## 2. Load Selected Companies

Loads `data/selected_companies.csv` produced by `01_explore_companies.ipynb`.

In [2]:
companies = pd.read_csv(DATA_DIR / "selected_companies.csv", dtype={"cik": str})
companies["cik"] = companies["cik"].str.zfill(10)

if MAX_COMPANIES:
    companies = companies.head(MAX_COMPANIES)

print(f"Loaded {len(companies)} companies")
print(f"\nBy sector:")
print(companies["sector"].value_counts().to_string())
companies.head()

Loaded 41 companies

By sector:
sector
Industrials               10
Consumer Discretionary     9
Real Estate                5
Financials                 4
Health Care                4
Communication Services     2
Consumer Staples           2
Information Technology     2
Energy                     1
Materials                  1
Utilities                  1


,ticker,company,sector,cik,total_10k,earliest_year,latest_year,error
0,LYV,Live Nation Entertainment,Communication Services,0001335258,14,2013.0,2026.0,NaN
1,SATS,EchoStar,Communication Services,0001415404,16,2011.0,2026.0,NaN
2,ULTA,Ulta Beauty,Consumer Discretionary,0001403568,17,2010.0,2026.0,NaN
3,LVS,Las Vegas Sands,Consumer Discretionary,0001300514,17,2010.0,2026.0,NaN
4,PHM,PulteGroup,Consumer Discretionary,0000822416,15,2012.0,2026.0,NaN


## 3. Helper Functions

In [3]:
SUBMISSIONS_API = "https://data.sec.gov/submissions/CIK{cik}.json"


def get_10k_filings(cik: str, start_year: int, end_year: int) -> pd.DataFrame:
    url = SUBMISSIONS_API.format(cik=cik)
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    data = response.json()
    time.sleep(REQUEST_DELAY)

    filings = data.get("filings", {}).get("recent", {})
    if not filings:
        return pd.DataFrame()

    df = pd.DataFrame({
        "form":             filings["form"],
        "filing_date":      filings["filingDate"],
        "accession_number": filings["accessionNumber"],
        "primary_document": filings["primaryDocument"],
    })

    df = df[df["form"] == "10-K"].copy()
    df["year"] = pd.to_datetime(df["filing_date"]).dt.year
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]
    return df.reset_index(drop=True)

In [4]:
FILING_BASE_URL = "https://www.sec.gov/Archives/edgar/data/{cik}/{accession}/{document}"


def download_filing(cik: str, accession_number: str, primary_document: str) -> str:
    accession_clean = accession_number.replace("-", "")
    cik_numeric = str(int(cik))
    url = FILING_BASE_URL.format(
        cik=cik_numeric, accession=accession_clean, document=primary_document
    )
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    time.sleep(REQUEST_DELAY)
    return response.text

In [5]:
ITEM_1A_PATTERN = re.compile(
    r'^[ \t]*item\s*1a[\s\W]{0,30}risk\s*factors[^\n]*\n(.+?)(?=^[ \t]*item\s*(?:1b|2)\b)',
    re.IGNORECASE | re.MULTILINE | re.DOTALL,
)
ITEM_1A_FALLBACK = re.compile(
    r'^[ \t]*item\s*1a[^\n]*\n(.{200,}?)(?=^[ \t]*item\s*(?:1b|2)\b)',
    re.IGNORECASE | re.MULTILINE | re.DOTALL,
)


def extract_item_1a(html: str) -> str | None:
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "table"]):
        tag.decompose()
    # Preserve paragraph breaks; only collapse horizontal whitespace
    text = soup.get_text(separator="\n")
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text).strip()
    match = ITEM_1A_PATTERN.search(text) or ITEM_1A_FALLBACK.search(text)
    return match.group(1).strip() if match else None

## 4. Run Full Pipeline

For each company: fetch filing list → download each 10-K → extract Item 1A → save.

- `data/item1a/{cik}_{year}.txt` — one file per company per year
- `data/filing_index.csv` — master index with extraction status

In [ ]:
def run_pipeline(
    companies: pd.DataFrame,
    start_year: int,
    end_year: int,
    force: bool = False,
) -> pd.DataFrame:
    records = []

    for _, company in tqdm(companies.iterrows(), total=len(companies), desc="Companies"):
        cik    = company["cik"]
        ticker = company["ticker"]
        name   = company["company"]

        try:
            filings = get_10k_filings(cik, start_year, end_year)
        except Exception as e:
            print(f"  Could not fetch filings for {ticker}: {e}")
            continue

        if filings.empty:
            print(f"  No 10-K filings found for {ticker} in range")
            continue

        for _, filing in filings.iterrows():
            year             = filing["year"]
            accession_number = filing["accession_number"]
            primary_document = filing["primary_document"]
            output_path      = DATA_DIR / "item1a" / f"{cik}_{year}.txt"

            if output_path.exists() and not force:
                records.append({"ticker": ticker, "company": name, "cik": cik,
                                 "year": year, "accession": accession_number,
                                 "status": "skipped (already exists)", "path": str(output_path)})
                continue

            try:
                html = download_filing(cik, accession_number, primary_document)
            except Exception as e:
                records.append({"ticker": ticker, "company": name, "cik": cik,
                                 "year": year, "accession": accession_number,
                                 "status": f"download_failed: {e}", "path": None})
                continue

            item_1a = extract_item_1a(html)
            if item_1a:
                output_path.write_text(item_1a, encoding="utf-8")
                status = "success"
            else:
                status = "item1a_not_found"

            records.append({"ticker": ticker, "company": name, "cik": cik,
                             "year": year, "accession": accession_number,
                             "status": status, "path": str(output_path) if item_1a else None})

    return pd.DataFrame(records)


# Set force=True to re-extract files that were already downloaded (e.g. after fixing extract_item_1a)
index = run_pipeline(companies, START_YEAR, END_YEAR, force = False)
index.to_csv(DATA_DIR / "filing_index.csv", index=False)

print(f"\nDone. Saved index to {DATA_DIR / 'filing_index.csv'}")
print(index["status"].value_counts())

Companies: 100%|██████████| 41/41 [09:43<00:00, 14.24s/it]


Done. Saved index to ../data/filing_index.csv
status
success             475
item1a_not_found    135
Name: count, dtype: int64


## 5. Inspect Results & Build Corpus

In [7]:
print("=== Pipeline Summary ===")
print(f"Total filings processed : {len(index)}")
print(f"Successful extractions  : {(index['status'] == 'success').sum()}")
print(f"Item 1A not found       : {(index['status'] == 'item1a_not_found').sum()}")
print(f"Download failures       : {index['status'].str.startswith('download').sum()}")
print(f"\nYear range covered      : {index['year'].min()} - {index['year'].max()}")
print(f"Unique companies        : {index['ticker'].nunique()}")

successful = index[index["status"] == "success"]
print("\n=== Successful Extractions Per Year ===")
print(successful.groupby("year").size().sort_index().to_string())

=== Pipeline Summary ===
Total filings processed : 610
Successful extractions  : 475
Item 1A not found       : 135
Download failures       : 0

Year range covered      : 2010 - 2026
Unique companies        : 41

=== Successful Extractions Per Year ===
year
2010     6
2011     6
2012    12
2013    19
2014    25
2015    28
2016    29
2017    29
2018    31
2019    33
2020    34
2021    38
2022    39
2023    39
2024    39
2025    38
2026    30


In [8]:
def load_corpus(filing_index: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in filing_index[filing_index["status"] == "success"].iterrows():
        text = Path(row["path"]).read_text(encoding="utf-8")
        rows.append({"ticker": row["ticker"], "company": row["company"],
                     "cik": row["cik"], "year": row["year"], "text": text})
    return pd.DataFrame(rows)


corpus = load_corpus(index)
corpus.to_csv(DATA_DIR / "corpus.csv", index=False)

print(f"Corpus saved to {DATA_DIR / 'corpus.csv'}")
print(f"Shape: {corpus.shape}")
corpus.head()

Corpus saved to ../data/corpus.csv
Shape: (475, 5)


,ticker,company,cik,year,text
0,LYV,Live Nation Entertainment,0001335258,2026,You should carefully consider each of the foll...
1,LYV,Live Nation Entertainment,0001335258,2025,You should carefully consider each of the foll...
2,LYV,Live Nation Entertainment,0001335258,2024,You should carefully consider each of the foll...
3,LYV,Live Nation Entertainment,0001335258,2023,You should carefully consider each of the foll...
4,LYV,Live Nation Entertainment,0001335258,2022,You should carefully consider each of the foll...


In [17]:
# In the next notebook, load the corpus with:
# corpus = pd.read_csv("../data/corpus.csv")